# Gross Margin Analysis — MarketCast Finance Dashboard

**Author:** Siham Chowdhury  
**Purpose:** Explore, validate and calculate gross margin from the Azure data warehouse. Documents the full methodology including fixed overhead allocation logic.

---

## What this notebook covers

1. Setup and connection
2. Understanding the core fact table
3. Revenue, COGS and Labour extraction
4. Fixed overhead allocations (BE, AE Synd, RTA)
5. Gross margin calculation
6. Validation against Excel source of truth
7. Interactive query function

---

## 1. Setup

Connect to the Azure PostgreSQL data warehouse. Credentials come from `.streamlit/secrets.toml` when running locally.

In [2]:
import pandas as pd
from pathlib import Path
from src.db.connection import get_engine

engine = get_engine()
engine.dispose()  # reset connection pool

QUERIES_DIR = Path("../src/queries")

print("Connected")

Connected


In [3]:
# Confirm the three distinct account_type_id values
pd.read_sql("""
    SELECT service_line_name,
        sub_service_line_name,
        COUNT(*) as row_count,
        SUM(amount) as total_amount
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1,2
    ORDER BY 1,2
""", engine)

,service_line_name,sub_service_line_name,row_count,total_amount
0,Ad Solutions,Ad Effect,6047,-1.464161e+07
1,Ad Solutions,Ad Solutions RTA,521,-2.251668e+06
2,Ad Solutions,Brand Effect,1586,-1.193832e+07
3,Ad Solutions,CA,14905,-2.019908e+07
4,Ad Solutions,Campaign Effect,1753,-4.339987e+06
5,Ad Solutions,Data Science,284,-2.146669e+06
6,Brand Tracking,Brand,7052,-1.636616e+07
7,Content,Custom,1570,-2.242973e+06
8,Custom,Custom,10410,-1.519284e+07
9,Data Science,Data Science,320,-1.688564e+06


---
## 2. Understanding the Core Fact Table

Everything flows from `core.rpt_project_revenue_and_costs`. Each row is a financial transaction line.

### The three row types

The `account_type_id` column tells you what kind of row it is:

| `account_type_id` | Meaning | How to use |
|---|---|---|
| `'Income'` | Revenue | Negate `amount` — credits are stored as negative |
| `'COGS'` | Cost of Sales | Sum `amount` directly |
| `NULL` | Labour / timesheet | Sum `amount` directly |

### Important: use `amount` not `amount_usd`
Both columns were verified to return identical totals for 2025 — `amount` is used throughout as it matches the Power Pivot model exactly.

### Key join — `dim_customers`
The fact table has `customer_id` which joins to `core.dim_customers` to get `top_level_parent_customer_name`. This is essential for grouping at parent client level (e.g. all Disney subsidiaries roll up to `Disney`).

In [2]:
# Confirm the three distinct account_type_id values
pd.read_sql("""
    SELECT 
        account_type_id,
        COUNT(*) as row_count,
        SUM(amount) as total_amount
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1
    ORDER BY 2 DESC
""", engine)

,account_type_id,row_count,total_amount
0,NaN,786520,3.457881e+07
1,COGS,39882,4.519082e+07
2,Income,6595,-1.356955e+08


In [3]:
# Show customer join — one customer_id can map to multiple subsidiaries
# All roll up to top_level_parent_customer_name
# Example: Disney has 3 customer_ids but one top-level parent
pd.read_sql("""
    SELECT 
        c.top_level_parent_customer_name,
        r.customer_id,
        c.customer_full_name,
        -SUM(r.amount) AS revenue_2025
    FROM core.rpt_project_revenue_and_costs r
    LEFT JOIN core.dim_customers c ON r.customer_id = c.customer_id
    WHERE r.account_type_id = 'Income'
      AND r.accounting_period_is_posted = TRUE
      AND EXTRACT(YEAR FROM r.accounting_period_start_date) = 2025
      AND c.top_level_parent_customer_name = 'Disney'
    GROUP BY 1,2,3
    ORDER BY 4 DESC
""", engine)

,top_level_parent_customer_name,customer_id,customer_full_name,revenue_2025
0,Disney,4355,Disney,7889375.39
1,Disney,4769,Disney : Hulu,2016575.00
2,Disney,141658,Disney : Disney Branded Television (DBT),828356.40
3,Disney,128877,Disney : Disney+,586580.00
4,Disney,44119,Disney : The Walt Disney Company,518962.50
5,Disney,124721,Disney : Searchlight Pictures,297762.75
6,Disney,92851,Disney : Walt Disney Company US,200042.00
7,Disney,93361,Disney : ABC,94345.00
8,Disney,5353,Disney : Disney Branded Television (DBT) : Dis...,49250.00


---
## 3. Revenue, COGS and Labour

Query `1_revenue_cogs_labour_by_period.sql` extracts all three metrics in a single pass using CASE WHEN.

### Verified totals for 2025
- Revenue: **$135.7M** ✅ matches Excel Flash - SL sheet exactly
- COGS: **$45.2M** ✅
- Labour: **$34.6M** ✅

In [4]:
# Load revenue/COGS/labour query
df_base = pd.read_sql(
    (QUERIES_DIR / "1_revenue_cogs_labour_by_period.sql").read_text(), 
    engine
)
df_base["accounting_period_start_date"] = pd.to_datetime(df_base["accounting_period_start_date"])

# 2025 totals
totals_2025 = df_base[df_base["accounting_period_start_date"].dt.year == 2025].agg({
    "revenue": "sum",
    "cogs": "sum",
    "labour": "sum"
}).round(0)

print("2025 totals:")
print(totals_2025)
print(f"\nRevenue: ${totals_2025['revenue']/1e6:.1f}M")
print(f"COGS:    ${totals_2025['cogs']/1e6:.1f}M")
print(f"Labour:  ${totals_2025['labour']/1e6:.1f}M")

2025 totals:
revenue    135695510.0
cogs        45190818.0
labour      34578815.0
dtype: float64

Revenue: $135.7M
COGS:    $45.2M
Labour:  $34.6M


---
## 4. Fixed Overhead Allocations

Three product lines have fixed yearly overhead costs that need to be allocated across customers proportionally to their revenue share.

### The three allocations


| Product | SQL filter | Fixed cost (2025) |
|---|---|---|
| Brand Effect | `sub_service_line_name = 'Brand Effect'` | $10,158,000 |
| AE Syndicated | `sub_service_line_name = 'Ad Effect'` | $938,000 |
| RTA | `sub_service_line_name = 'Ad Solutions RTA'` | $833,000 |

### The formula

```
Monthly allocation per customer =
    (Customer annual product revenue / Total annual product revenue) 
    × Fixed yearly cost 
    ÷ 12
```

### Worked example — Disney Brand Effect 2025

- Total BE revenue across all customers (2025): **$22,873,199**
- Disney TV BE revenue (2025): **$518,963**
- Disney TV share: $518,963 / $22,873,199 = **2.27%**
- Annual allocation: 2.27% × $10,158,000 = **$230,428**
- Monthly allocation: $230,428 / 12 = **$19,202 per month**

### Key design decisions

**1. Group at `top_level_parent_customer_name` level, not `customer_id`**  
Disney has 3 customer IDs (Disney: The Walt Disney Company, Disney, Disney: Hulu). If we group by `customer_id`, each subsidiary gets its own allocation row — causing duplicates. By joining to `dim_customers` and grouping by `top_level_parent_customer_name`, all Disney subsidiaries are consolidated into one Disney row.

**2. Fan out to all 12 months (Option A)**  
The fixed overhead exists regardless of whether a customer had activity in a given month. So the allocation is applied across all 12 months of the year, not just months where the customer had revenue. This ensures the full fixed cost is recovered each year.

**3. No rounding at row level**  
Rounding each monthly allocation to 2dp accumulates penny-level errors. Division is kept at full precision and rounding only applied in display.

**4. FULL OUTER JOIN in the final query**  
Some customers have allocation rows but no matching revenue rows in `base` (e.g. they had Brand Effect revenue in prior periods). A LEFT JOIN from `base` would drop these. FULL OUTER JOIN ensures all allocation rows are preserved.

In [5]:
# Worked example — Disney Brand Effect 2025
df_be = pd.read_sql(
    (QUERIES_DIR / "2_be_allocation.sql").read_text(),
    engine
)
df_be["accounting_period_start_date"] = pd.to_datetime(df_be["accounting_period_start_date"])

disney_2025 = df_be[
    (df_be["top_level_parent_customer_name"] == "Disney") &
    (df_be["yr"] == 2025)
]

print("Disney 2025 — BE Allocation by vertical:")
print(
    disney_2025
    .groupby("vertical_name")
    .agg(
        annual_be_rev=("annual_be_rev", "first"),
        monthly_allocation=("be_allocation_per_month", "first"),
        months=("be_allocation_per_month", "count"),
        total_allocated=("be_allocation_per_month", "sum")
    )
    .round(2)
)

# Manual check
tot_be = disney_2025["annual_tot_be_rev"].iloc[0]
fixed = disney_2025["fixed_cost_be"].iloc[0]
disney_tv = disney_2025[disney_2025["vertical_name"] == "TV"]["annual_be_rev"].iloc[0]

print(f"\n=== Manual check — Disney TV 2025 ===")
print(f"Disney TV annual BE rev:         ${disney_tv:,.0f}")
print(f"Total BE rev all customers:      ${tot_be:,.0f}")
print(f"Disney TV share:                 {disney_tv/tot_be*100:.2f}%")
print(f"Fixed cost:                      ${fixed:,.0f}")
print(f"Annual allocation:               ${disney_tv/tot_be*fixed:,.0f}")
print(f"Monthly allocation (/ 12):       ${disney_tv/tot_be*fixed/12:,.2f}")

Disney 2025 — BE Allocation by vertical:
               annual_be_rev  monthly_allocation  months  total_allocated
vertical_name                                                            
Streaming            40000.0                0.15      12             1.80
TV                 1018212.5                3.71      12            44.52

=== Manual check — Disney TV 2025 ===
Disney TV annual BE rev:         $1,018,212
Total BE rev all customers:      $22,873,199
Disney TV share:                 4.45%
Fixed cost:                      $1,000
Annual allocation:               $45
Monthly allocation (/ 12):       $3.71


In [6]:
# Sanity check — annual sum of all monthly allocations should equal fixed cost per year
print("BE allocation annual sum (should equal fixed cost per year):")
print(df_be.groupby("yr")["be_allocation_per_month"].sum().round(0))

df_ae = pd.read_sql((QUERIES_DIR / "3_ae_synd_allocation.sql").read_text(), engine)
print("\nAE allocation annual sum:")
print(df_ae.groupby("yr")["ae_allocation_per_month"].sum().round(0))

df_rta = pd.read_sql((QUERIES_DIR / "4_rta_allocation.sql").read_text(), engine)
print("\nRTA allocation annual sum:")
print(df_rta.groupby("yr")["rta_allocation_per_month"].sum().round(0))

BE allocation annual sum (should equal fixed cost per year):
yr
2022    1000.0
2023    1000.0
2024    1000.0
2025    1000.0
2026    1000.0
Name: be_allocation_per_month, dtype: float64

AE allocation annual sum:
yr
2022    1000.0
2023    1000.0
2024    1000.0
2025    1000.0
2026    1000.0
Name: ae_allocation_per_month, dtype: float64

RTA allocation annual sum:
yr
2022    1000.0
2023    1000.0
2024    1000.0
2025    1000.0
2026    1000.0
Name: rta_allocation_per_month, dtype: float64


---
## 5. Gross Margin Calculation

Query `5_gross_margin.sql` joins everything together and calculates:

```
Gross Margin = Revenue - COGS - BE Allocation - AE Allocation - RTA Allocation
GM% = Gross Margin / Revenue × 100
```

Note: Labour is included in the output but **not** deducted from Gross Margin. It is used separately for the Contribution Margin calculation (GM - Labour = Contribution $).

### Fixed costs used (confirm with Erik for future years)

| Year | BE | AE Synd | RTA |
|---|---|---|---|
| 2022-2024 | $1,015,000 | $938,000 | $833,000 |
| 2025 | $10,158,000 | $938,000 | $833,000 |
| 2026 | $1,015,000 | $938,000 | $833,000 |

In [7]:
# Load the full gross margin query
df_gm = pd.read_sql(
    (QUERIES_DIR / "5_gross_margin.sql").read_text(),
    engine
)
df_gm["accounting_period_start_date"] = pd.to_datetime(df_gm["accounting_period_start_date"])

print(f"Total rows: {len(df_gm):,}")
print(f"Columns: {list(df_gm.columns)}")
df_gm.head(5)

Total rows: 38,809
Columns: ['accounting_period_name', 'accounting_period_start_date', 'top_level_parent_customer_name', 'vertical_name', 'service_line_name', 'sub_service_line_name', 'revenue', 'cogs', 'labour', 'be_allocation', 'ae_allocation', 'rta_allocation', 'gross_margin', 'gm_pct']


,accounting_period_name,accounting_period_start_date,top_level_parent_customer_name,vertical_name,service_line_name,sub_service_line_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,gm_pct
0,Jan 2017,2017-01-01,Unassigned,Client Advisor,NaN,NaN,148946.00,74872.39,0.0,0.0,0.0,0.0,74073.61,49.7
1,Jan 2017,2017-01-01,Unassigned,NaN,NaN,NaN,3162361.13,977157.94,0.0,0.0,0.0,0.0,2185203.19,69.1
2,Feb 2017,2017-02-01,Unassigned,Client Advisor,NaN,NaN,88140.95,50448.21,0.0,0.0,0.0,0.0,37692.74,42.8
3,Feb 2017,2017-02-01,Unassigned,NaN,NaN,NaN,2594102.14,555997.88,0.0,0.0,0.0,0.0,2038104.26,78.6
4,Mar 2017,2017-03-01,Unassigned,Client Advisor,NaN,NaN,75917.00,51972.68,0.0,0.0,0.0,0.0,23944.32,31.5


In [8]:
# 2025 totals across all customers
totals_gm = df_gm[
    df_gm["accounting_period_start_date"].dt.year == 2025
].agg({
    "revenue": "sum",
    "cogs": "sum",
    "labour": "sum",
    "be_allocation": "sum",
    "ae_allocation": "sum",
    "rta_allocation": "sum",
    "gross_margin": "sum"
}).round(0)

print("2025 totals:")
print(totals_gm)
print(f"\nGM%: {totals_gm['gross_margin']/totals_gm['revenue']*100:.1f}%")
print(f"Contribution $ (GM - Labour): ${(totals_gm['gross_margin']-totals_gm['labour'])/1e6:.1f}M")
print(f"CM%: {(totals_gm['gross_margin']-totals_gm['labour'])/totals_gm['revenue']*100:.1f}%")

2025 totals:
revenue           135695510.0
cogs               45190818.0
labour             34578815.0
be_allocation      10158000.0
ae_allocation        938000.0
rta_allocation       833000.0
gross_margin       78575692.0
dtype: float64

GM%: 57.9%
Contribution $ (GM - Labour): $44.0M
CM%: 32.4%


---
## 6. Validation Against Excel

Cross-checking key rows from the Excel Contribution sheet against our SQL output.

From the Excel (2025, all clients):

| Sub Service Line | Revenue ($k) | COGS ($k) | BE Alloc ($k) | AE Alloc ($k) | RTA Alloc ($k) | GM ($k) | GM% |
|---|---|---|---|---|---|---|---|
| Ad Effect | 17,687 | 3,045 | — | 938 | — | 13,704 | 77% |
| Ad Solutions RTA | 3,226 | 974 | — | — | 833 | 1,419 | 44% |
| Brand Effect | 22,873 | 10,935 | 10,158 | — | — | 1,780 | 8% |
| Grand Total | 105,091 | 34,786 | 10,158 | 938 | 833 | 58,377 | 56% |

> Note: The Excel Grand Total ($105M) is lower than our SQL total ($135.7M) because the Excel pivot has filters applied to specific clients/accounts. The individual product line numbers (Brand Effect etc.) should match exactly.

In [9]:
# Validate key sub service lines against Excel
check_lines = ["Brand Effect", "Ad Effect", "Ad Solutions RTA"]

df_check = df_gm[
    (df_gm["accounting_period_start_date"].dt.year == 2025) &
    (df_gm["sub_service_line_name"].isin(check_lines))
].groupby("sub_service_line_name").agg({
    "revenue": "sum",
    "cogs": "sum",
    "be_allocation": "sum",
    "ae_allocation": "sum",
    "rta_allocation": "sum",
    "gross_margin": "sum"
}).round(0)

df_check["gm_pct"] = (df_check["gross_margin"] / df_check["revenue"] * 100).round(1)

# Convert to $000s to match Excel
for col in ["revenue","cogs","be_allocation","ae_allocation","rta_allocation","gross_margin"]:
    df_check[col] = (df_check[col] / 1000).round(0)

print("SQL output vs Excel (values in $000s):")
print(df_check.to_string())

SQL output vs Excel (values in $000s):
                       revenue     cogs  be_allocation  ae_allocation  rta_allocation  gross_margin  gm_pct
sub_service_line_name                                                                                      
Ad Effect              17687.0   3045.0            0.0          938.0             0.0       13704.0    77.5
Ad Solutions RTA        3226.0    974.0            0.0            0.0           833.0        1419.0    44.0
Brand Effect           22873.0  10935.0        10158.0            0.0             0.0        1780.0     7.8


---
## 7. Interactive Query Function

Use this function to slice the gross margin data by any combination of dimensions and get a clean metrics summary.

In [10]:
def get_metrics(
    year: int = 2025,
    sub_service_line: str = None,
    service_line: str = None,
    customer: str = None,
    vertical: str = None,
    group_by: list = None
) -> pd.DataFrame:
    """
    Slice gross margin data by any combination of dimensions.

    Parameters
    ----------
    year : int
        Accounting year to filter to (default 2025)
    sub_service_line : str, optional
        Filter to a specific sub service line (e.g. 'Brand Effect')
    service_line : str, optional
        Filter to a specific service line (e.g. 'Ad Solutions')
    customer : str, optional
        Filter to a specific top level parent customer (e.g. 'Disney')
    vertical : str, optional
        Filter to a specific vertical (e.g. 'TV')
    group_by : list, optional
        Columns to group by in output. Defaults to ['sub_service_line_name'].
        Options: 'service_line_name', 'sub_service_line_name', 
                 'top_level_parent_customer_name', 'vertical_name',
                 'accounting_period_name'

    Returns
    -------
    pd.DataFrame with columns:
        revenue, cogs, labour, be_allocation, ae_allocation, rta_allocation,
        gross_margin, gm_pct, contribution, cm_pct

    Examples
    --------
    # All metrics for Brand Effect in 2025
    get_metrics(year=2025, sub_service_line='Brand Effect')

    # Disney metrics by vertical
    get_metrics(year=2025, customer='Disney', group_by=['vertical_name'])

    # Monthly revenue trend for Ad Solutions
    get_metrics(year=2025, service_line='Ad Solutions',
                group_by=['accounting_period_name'])
    """
    if group_by is None:
        group_by = ["sub_service_line_name"]

    # Apply filters
    mask = df_gm["accounting_period_start_date"].dt.year == year

    if sub_service_line:
        mask &= df_gm["sub_service_line_name"] == sub_service_line
    if service_line:
        mask &= df_gm["service_line_name"] == service_line
    if customer:
        mask &= df_gm["top_level_parent_customer_name"] == customer
    if vertical:
        mask &= df_gm["vertical_name"] == vertical

    df = df_gm[mask].copy()

    if df.empty:
        print("No data found for the given parameters.")
        return pd.DataFrame()

    # Aggregate
    agg_cols = {
        "revenue": "sum",
        "cogs": "sum",
        "labour": "sum",
        "be_allocation": "sum",
        "ae_allocation": "sum",
        "rta_allocation": "sum",
        "gross_margin": "sum"
    }

    result = df.groupby(group_by).agg(agg_cols).reset_index()

    # Derived metrics
    result["gm_pct"] = (result["gross_margin"] / result["revenue"].replace(0, float("nan")) * 100).round(1)
    result["contribution"] = result["gross_margin"] - result["labour"]
    result["cm_pct"] = (result["contribution"] / result["revenue"].replace(0, float("nan")) * 100).round(1)

    # Round financials
    for col in ["revenue","cogs","labour","be_allocation","ae_allocation","rta_allocation","gross_margin","contribution"]:
        result[col] = result[col].round(0)

    # Sort by revenue descending
    result = result.sort_values("revenue", ascending=False)

    return result

print("get_metrics() function loaded")

get_metrics() function loaded


### Example queries

In [11]:
# Example 1: All metrics by sub service line for 2025
print("=== All sub service lines — 2025 ===")
get_metrics(year=2025, group_by=["service_line_name", "sub_service_line_name"])

=== All sub service lines — 2025 ===


,service_line_name,sub_service_line_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,gm_pct,contribution,cm_pct
3,Ad Solutions,CA,30604373.0,10405289.0,0.0,0.0,0.0,0.0,20199084.0,66.0,20199084.0,66.0
8,Custom,Custom,24618216.0,9425375.0,0.0,0.0,0.0,0.0,15192841.0,61.7,15192841.0,61.7
6,Brand Tracking,Brand,23498047.0,7131892.0,0.0,0.0,0.0,0.0,16366155.0,69.6,16366155.0,69.6
2,Ad Solutions,Brand Effect,22873199.0,10934876.0,0.0,10158000.0,0.0,0.0,1780322.0,7.8,1780322.0,7.8
0,Ad Solutions,Ad Effect,17687061.0,3045452.0,0.0,0.0,938000.0,0.0,13703609.0,77.5,13703609.0,77.5
4,Ad Solutions,Campaign Effect,5876676.0,1536688.0,0.0,0.0,0.0,0.0,4339987.0,73.9,4339987.0,73.9
1,Ad Solutions,Ad Solutions RTA,3225533.0,973865.0,0.0,0.0,0.0,833000.0,1418668.0,44.0,1418668.0,44.0
7,Content,Custom,3133214.0,890241.0,0.0,0.0,0.0,0.0,2242973.0,71.6,2242973.0,71.6
5,Ad Solutions,Data Science,2852038.0,705370.0,0.0,0.0,0.0,0.0,2146669.0,75.3,2146669.0,75.3
9,Data Science,Data Science,2161222.0,472658.0,0.0,0.0,0.0,0.0,1688564.0,78.1,1688564.0,78.1


In [12]:
# Example 2: Brand Effect only — breakdown by customer
print("=== Brand Effect 2025 — top 10 customers ===")
get_metrics(
    year=2025,
    sub_service_line="Brand Effect",
    group_by=["top_level_parent_customer_name"]
).head(10)

=== Brand Effect 2025 — top 10 customers ===


,top_level_parent_customer_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,gm_pct,contribution,cm_pct
30,NBCUniversal,4833350.0,29209.0,0.0,2146493.0,0.0,0.0,2657648.0,55.0,2657648.0,55.0
15,Fox,1828013.0,143.0,0.0,811822.0,0.0,0.0,1016048.0,55.6,1016048.0,55.6
35,Nissan Motor Corp,1420069.0,0.0,0.0,630653.0,0.0,0.0,789415.0,55.6,789415.0,55.6
10,DirecTV,1225000.0,0.0,0.0,544023.0,0.0,0.0,680977.0,55.6,680977.0,55.6
52,Walmart,1085062.0,5680.0,0.0,481877.0,0.0,0.0,597506.0,55.1,597506.0,55.1
44,State Farm,1076910.0,2537.0,0.0,478256.0,0.0,0.0,596117.0,55.4,596117.0,55.4
11,Disney,1058212.0,0.0,0.0,469953.0,0.0,0.0,588260.0,55.6,588260.0,55.6
5,Capital One,896000.0,0.0,0.0,397914.0,0.0,0.0,498086.0,55.6,498086.0,55.6
21,Intuit,823717.0,1923.0,0.0,365813.0,0.0,0.0,455981.0,55.4,455981.0,55.4
29,Miller Coors,740040.0,3583.0,0.0,328652.0,0.0,0.0,407804.0,55.1,407804.0,55.1


In [13]:
# Example 3: Disney — all metrics by sub service line
print("=== Disney 2025 — by sub service line ===")
get_metrics(
    year=2025,
    customer="Disney",
    group_by=["service_line_name", "sub_service_line_name"]
)

=== Disney 2025 — by sub service line ===


,service_line_name,sub_service_line_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,gm_pct,contribution,cm_pct
3,Ad Solutions,CA,7988750.0,3147242.0,0.0,0.0,0.0,0.0,4841507.0,60.6,4841507.0,60.6
5,Content,Custom,1353470.0,335013.0,0.0,0.0,0.0,0.0,1018457.0,75.2,1018457.0,75.2
2,Ad Solutions,Brand Effect,1058212.0,0.0,0.0,469953.0,0.0,0.0,588260.0,55.6,588260.0,55.6
0,Ad Solutions,Ad Effect,895700.0,165250.0,0.0,0.0,47502.0,0.0,682948.0,76.2,682948.0,76.2
4,Brand Tracking,Brand,598097.0,479623.0,0.0,0.0,0.0,0.0,118474.0,19.8,118474.0,19.8
6,Custom,Custom,547000.0,154050.0,0.0,0.0,0.0,0.0,392950.0,71.8,392950.0,71.8
1,Ad Solutions,Ad Solutions RTA,40020.0,0.0,0.0,0.0,0.0,10335.0,29685.0,74.2,29685.0,74.2


In [14]:
# Example 4: Monthly revenue trend for Ad Solutions in 2025
print("=== Ad Solutions 2025 — monthly trend ===")
get_metrics(
    year=2025,
    service_line="Ad Solutions",
    group_by=["accounting_period_name"]
)

=== Ad Solutions 2025 — monthly trend ===


,accounting_period_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,gm_pct,contribution,cm_pct
3,Feb 2025,7971310.0,2502249.0,0.0,846500.0,78167.0,69417.0,4474978.0,56.1,4474978.0,56.1
8,May 2025,7749892.0,2847541.0,0.0,846500.0,78167.0,69417.0,3908267.0,50.4,3908267.0,50.4
0,Apr 2025,7554633.0,2730455.0,0.0,846500.0,78167.0,69417.0,3830094.0,50.7,3830094.0,50.7
9,Nov 2025,7115513.0,2261768.0,0.0,846500.0,78167.0,69417.0,3859662.0,54.2,3859662.0,54.2
1,Aug 2025,7091759.0,2169432.0,0.0,846500.0,78167.0,69417.0,3928243.0,55.4,3928243.0,55.4
6,Jun 2025,6917613.0,2131327.0,0.0,846500.0,78167.0,69417.0,3792203.0,54.8,3792203.0,54.8
7,Mar 2025,6723208.0,2428753.0,0.0,846500.0,78167.0,69417.0,3300371.0,49.1,3300371.0,49.1
4,Jan 2025,6662080.0,1978637.0,0.0,846500.0,78167.0,69417.0,3689360.0,55.4,3689360.0,55.4
5,Jul 2025,6476723.0,2307214.0,0.0,846500.0,78167.0,69417.0,3175426.0,49.0,3175426.0,49.0
11,Sep 2025,6425624.0,2381618.0,0.0,846500.0,78167.0,69417.0,3049922.0,47.5,3049922.0,47.5


In [15]:
# Example 5: NBCUniversal Brand Effect — by vertical and year
print("=== NBCUniversal — Brand Effect 2024 ===")
get_metrics(
    year=2024,
    customer="NBCUniversal",
    sub_service_line="Brand Effect",
    group_by=["vertical_name"]
)

=== NBCUniversal — Brand Effect 2024 ===


,vertical_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,gm_pct,contribution,cm_pct
1,TV,4670004.0,24287.0,0.0,182123.0,0.0,0.0,4463594.0,95.6,4463594.0,95.6
0,Streaming,2000.0,0.0,0.0,78.0,0.0,0.0,1922.0,96.1,1922.0,96.1


In [16]:
# Example 6: Top 10 clients by revenue in 2025
print("=== Top 10 clients by revenue — 2025 ===")
get_metrics(
    year=2025,
    group_by=["top_level_parent_customer_name"]
).head(10)

=== Top 10 clients by revenue — 2025 ===


,top_level_parent_customer_name,revenue,cogs,labour,be_allocation,ae_allocation,rta_allocation,gross_margin,gm_pct,contribution,cm_pct
107,NBCUniversal,16297212.0,3695813.0,612537.0,2146493.0,0.0,283011.0,10171895.0,62.4,9559358.0,58.7
50,Disney,12481249.0,4270827.0,1161402.0,469953.0,47502.0,10335.0,7682632.0,61.6,6521230.0,52.2
102,Meta,9400637.0,3107057.0,1551337.0,0.0,0.0,20660.0,6272920.0,66.7,4721583.0,50.2
61,Ford Motor Corp,7216524.0,1143445.0,1400938.0,88820.0,224541.0,0.0,5759718.0,79.8,4358780.0,60.4
90,LEGO,6533259.0,2992289.0,1362246.0,0.0,0.0,0.0,3540969.0,54.2,2178723.0,33.3
15,Amazon,5311251.0,1445120.0,854510.0,0.0,0.0,202143.0,3663989.0,69.0,2809478.0,52.9
71,Google,4136333.0,1016056.0,877795.0,26646.0,1591.0,0.0,3092041.0,74.8,2214246.0,53.5
121,Paramount Global,3931904.0,1061353.0,1112212.0,135229.0,0.0,9975.0,2725347.0,69.3,1613135.0,41.0
182,Walmart,3393091.0,572078.0,958060.0,481877.0,40693.0,0.0,2298443.0,67.7,1340383.0,39.5
34,Chase,3387886.0,608496.0,1022749.0,0.0,169956.0,10002.0,2599431.0,76.7,1576682.0,46.5


In [17]:
# How much labour exists vs what's being captured in df_gm
print("Total labour in df_gm 2025:")
print(df_gm[df_gm["accounting_period_start_date"].dt.year == 2025]["labour"].sum().round(0))

print("\nLabour by service line:")
print(
    df_gm[df_gm["accounting_period_start_date"].dt.year == 2025]
    .groupby("service_line_name")["labour"]
    .sum()
    .round(0)
    .sort_values(ascending=False)
)

Total labour in df_gm 2025:
34578815.0

Labour by service line:
service_line_name
Ad Solutions      0.0
Brand Tracking    0.0
Content           0.0
Custom            0.0
Data Science      0.0
Name: labour, dtype: float64


In [18]:
# Check what account_type_id values exist for labour rows
# and whether NULL is being captured correctly

pd.read_sql("""
    SELECT
        account_type_id,
        is_part_of_income_statement,
        COUNT(*) as rows,
        SUM(amount) as total_amount
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1, 2
    ORDER BY 3 DESC
""", engine)

,account_type_id,is_part_of_income_statement,rows,total_amount
0,NaN,False,786520,3.457881e+07
1,COGS,True,39882,4.519082e+07
2,Income,True,6572,-1.356955e+08
3,Income,False,23,-1.455192e-11


In [19]:
# Check what sub_service_line_name looks like on labour rows
pd.read_sql("""
    SELECT
        sub_service_line_name,
        service_line_name,
        COUNT(*) as rows,
        SUM(amount) as labour_amount
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
    AND account_type_id IS NULL
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1, 2
    ORDER BY 3 DESC
    LIMIT 10
""", engine)

,sub_service_line_name,service_line_name,rows,labour_amount
0,None,None,786520,3.457881e+07


In [20]:
# Check labour on blank/null sub service line rows
df_blank = df_gm[
    (df_gm["accounting_period_start_date"].dt.year == 2025) &
    (df_gm["sub_service_line_name"].isna())
].agg({
    "revenue": "sum",
    "cogs": "sum",
    "labour": "sum",
    "gross_margin": "sum"
}).round(0)

print("Blank sub service line rows 2025:")
print(df_blank)

Blank sub service line rows 2025:
revenue          -834068.0
cogs             -330888.0
labour          34578815.0
gross_margin     -503180.0
dtype: float64


In [21]:
# What's in the blank sub service line revenue rows?
pd.read_sql("""
    SELECT
        account_type_id,
        account_name,
        COUNT(*) as rows,
        SUM(amount) as total_amount
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    AND sub_service_line_name IS NULL
    AND account_type_id = 'Income'
    GROUP BY 1, 2
    ORDER BY 3 DESC
    LIMIT 10
""", engine)

,account_type_id,account_name,rows,total_amount
0,Income,Revenues : Customer Discounts,513,834067.93


In [22]:
# Replicate the Excel Contribution sheet view
# Group by service_line_name and sub_service_line_name
# Filter to 2025, positive revenue, non-Unassigned customers

df_contrib = df_gm[
    (df_gm["accounting_period_start_date"].dt.year == 2025) &
    (df_gm["top_level_parent_customer_name"] != "Unassigned")
].groupby(
    ["service_line_name", "sub_service_line_name"],
    dropna=False
).agg({
    "revenue":        "sum",
    "cogs":           "sum",
    "be_allocation":  "sum",
    "ae_allocation":  "sum",
    "rta_allocation": "sum",
    "gross_margin":   "sum",
    "labour":         "sum"
}).reset_index()

# Derived metrics
df_contrib["gm_pct"]      = (df_contrib["gross_margin"] / df_contrib["revenue"].replace(0, float("nan")) * 100).round(1)
df_contrib["contribution"] = df_contrib["gross_margin"] - df_contrib["labour"]
df_contrib["cm_pct"]      = (df_contrib["contribution"] / df_contrib["revenue"].replace(0, float("nan")) * 100).round(1)

# Convert to $000s to match Excel
for col in ["revenue","cogs","be_allocation","ae_allocation","rta_allocation","gross_margin","labour","contribution"]:
    df_contrib[col] = (df_contrib[col] / 1000).round(0)

# Grand total row
grand = df_contrib[[
    "revenue","cogs","be_allocation","ae_allocation",
    "rta_allocation","gross_margin","labour","contribution"
]].sum().to_frame().T
grand["service_line_name"] = "Grand Total"
grand["sub_service_line_name"] = ""
grand["gm_pct"] = (grand["gross_margin"] / grand["revenue"] * 100).round(1)
grand["cm_pct"] = (grand["contribution"] / grand["revenue"] * 100).round(1)

result = pd.concat([df_contrib, grand], ignore_index=True)

# Display all columns matching Excel
print(result[[
    "service_line_name", "sub_service_line_name",
    "revenue", "cogs", "be_allocation", "ae_allocation", "rta_allocation",
    "gross_margin", "gm_pct", "labour", "contribution", "cm_pct"
]].to_string(index=False))

service_line_name sub_service_line_name  revenue    cogs  be_allocation  ae_allocation  rta_allocation  gross_margin  gm_pct  labour  contribution  cm_pct
     Ad Solutions             Ad Effect  17687.0  3012.0            0.0          938.0             0.0       13737.0    77.7     0.0       13737.0    77.7
     Ad Solutions      Ad Solutions RTA   3226.0   974.0            0.0            0.0           833.0        1419.0    44.0     0.0        1419.0    44.0
     Ad Solutions          Brand Effect  22873.0 10934.0        10158.0            0.0             0.0        1781.0     7.8     0.0        1781.0     7.8
     Ad Solutions                    CA  30604.0 10385.0            0.0            0.0             0.0       20219.0    66.1     0.0       20219.0    66.1
     Ad Solutions       Campaign Effect   5877.0  1536.0            0.0            0.0             0.0        4340.0    73.9     0.0        4340.0    73.9
     Ad Solutions          Data Science   2852.0   705.0            0.

In [23]:
# Rename DB values to match Excel labels
rename_ssl = {
    "CA": "Attribution",
}
df_contrib["sub_service_line_name"] = df_contrib["sub_service_line_name"].replace(rename_ssl)

In [1]:
# Confirm the three distinct account_type_id values
pd.read_sql("""
    SELECT service_line_name,
        sub_service_line_name,
        COUNT(*) as row_count,
        SUM(amount) as total_amount
    FROM core.rpt_project_revenue_and_costs
    WHERE accounting_period_is_posted = TRUE
    AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1,2
    ORDER BY 1,2
""", engine)

NameError: name 'pd' is not defined